# 🔑 INSTRUCTOR ANSWER KEY — do not distribute

# ⚛️ Microproject 5: Periodic Trends
*Elements of Data Science — Regression Microprojects* <br>

**Electronegativity** measures how strongly an atom pulls on electrons in a bond. **First ionization energy** is the energy needed to remove an atom's outermost electron. Both are classic periodic trends taught in general chemistry.

> **Your research question:** How strongly is first ionization energy associated with electronegativity across the periodic table — and which elements break the pattern?

In Lab 8 you built a **regression toolbox** step by step. This microproject gives you much less scaffolding: you decide how to explore, you build the line, and you judge whether it is any good. Your final product is **one publication-quality figure**, submitted to the class gallery, that your instructor will show in class.

**Review:** [Ch. 15.2 The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html), [Ch. 15.5 Visual Diagnostics](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html), [LibreTexts: Periodic Trends](https://chem.libretexts.org/Bookshelves/Inorganic_Chemistry/Supplemental_Modules_and_Websites_(Inorganic_Chemistry)/Descriptive_Chemistry/Periodic_Trends_of_Elemental_Properties/Periodic_Trends)

|Part|What you do|
|---|---|
|1|Rebuild your regression toolbox|
|2|Meet the data|
|3|Look first: scatter plot and a guess|
|4|Fit the line and interpret it|
|5|Check the fit: errors and `np.polyfit`|
|6|The twist: dig deeper|
|7|Make and submit your final figure|

## ⚙️ Initialization — DO NOT EDIT
👉 **Run the cell below first** (Shift+Enter) before anything else.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
from microproject_tools import my_project, submit_plot

### 🧰 Standard imports

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Enter your name (first and last), then check that this is the project you were assigned.

In [ ]:
name = 'Test Student'
my_project(name)

## 1. Your Regression Toolbox
Copy your four functions from **Lab 8, Questions 1.5 and 4.4** into the cell below (or rebuild them from [Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)). Then run the test cell: a perfect line $y = 2x + 1$ must give $r = 1$, slope 2, intercept 1.

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

In [ ]:
# Toolbox test — don't change this cell
test = Table().with_columns('x', make_array(1, 2, 3, 4), 'y', make_array(3, 5, 7, 9))
print('r =', correlation(test, 'x', 'y'), '  slope =', slope(test, 'x', 'y'), '  intercept =', intercept(test, 'x', 'y'))
print('✅ Toolbox works!' if np.isclose(correlation(test, 'x', 'y'), 1) and np.isclose(slope(test, 'x', 'y'), 2)
      and np.isclose(intercept(test, 'x', 'y'), 1) else '❌ Check your functions against Lab 8.')

## 2. Meet the Data
One row per element (elements 1–86 that have a Pauling electronegativity):
- **symbol**, **name**, **atomic number**
- **period** (row) and **group** (column, 1–18) in the periodic table
- **category**: e.g. alkali metal, transition metal, halogen
- **electronegativity**: Pauling scale (no units)
- **ionization energy (kJ/mol)**: first ionization energy

*Source:* [Periodic-Table-JSON](https://github.com/Bowserinator/Periodic-Table-JSON) (CC BY-SA), compiled from standard reference values.

In [ ]:
data = Table.read_table('elements.csv')
data

<font color=blue>**Explore.**</font> Which element has the highest electronegativity? The highest ionization energy? Which categories have the most elements? Use any Table methods you know (`num_rows`, `sort`, `where`, `group`, `hist`...). Add more cells if you like.

In [ ]:
print(data.num_rows, 'rows')
print('x range:', min(data.column('electronegativity')), 'to', max(data.column('electronegativity')))
print('y range:', min(data.column('ionization energy (kJ/mol)')), 'to', max(data.column('ionization energy (kJ/mol)')))

## 3. Look First
<font color=blue>**Question 3.1.**</font> Make a scatter plot with **`electronegativity`** on the horizontal axis and **`ionization energy (kJ/mol)`** on the vertical axis. No line yet.

In [ ]:
data.scatter('electronegativity', 'ionization energy (kJ/mol)')
plt.show()

<font color=green>Describe the **direction**, **form**, and **strength** of the association. Then guess the value of *r*. Which corner of the plot do the alkali metals sit in?</font>

---

ANSWER

---

## 4. Fit the Line
<font color=blue>**Question 4.1.**</font> Use your toolbox to compute `my_r`, `my_slope`, and `my_intercept` for predicting **ionization energy (kJ/mol)** from **electronegativity**.

In [ ]:
my_r = correlation(data, 'electronegativity', 'ionization energy (kJ/mol)')
my_slope = slope(data, 'electronegativity', 'ionization energy (kJ/mol)')
my_intercept = intercept(data, 'electronegativity', 'ionization energy (kJ/mol)')
print(f'r = {my_r:.3f}')
print(f'ionization energy (kJ/mol) = {my_slope:.4f} × electronegativity + {my_intercept:.2f}')

<font color=green>Interpret your slope **in words, with units**: "For each additional ___ unit of electronegativity, the ionization energy (kJ/mol) is predicted to change by ___ kJ/mol." How close was your guess for *r*?</font>

---

ANSWER

---

<font color=blue>**Question 4.2.**</font> Predict the first ionization energy of an element with electronegativity **2.5**. (Carbon has 2.55 — look up its actual value in the table and compare.) Store your prediction as `my_prediction`.

In [ ]:
my_prediction = my_slope * 2.5 + my_intercept
my_prediction

## 5. Check the Fit
A line can have a large *r* and still be the wrong model. Plot the **errors** (residuals), `actual y − predicted y`, against $x$. A good straight-line fit leaves errors that look like a patternless cloud centered on 0 ([Ch. 15.5](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)).

<font color=blue>**Question 5.1.**</font> Compute the array `errors` and make a scatter plot of the errors against $x$, with a horizontal line at 0 (`plt.axhline(0)`).

<details><summary><b>💡 Hint</b></summary>

`errors = data.column(y) - (my_slope * data.column(x) + my_intercept)` — then `plt.scatter(...)` with the x column and `errors`. This is the `line_errors` idea from Lab 8, Question 2.3.

</details>

In [ ]:
x = data.column('electronegativity')
y = data.column('ionization energy (kJ/mol)')
errors = y - (my_slope * x + my_intercept)
plt.scatter(x, errors, s=12)
plt.axhline(0, color='red')
plt.xlabel('electronegativity'); plt.ylabel('error')
plt.show()

<font color=green>Do the errors look like a patternless cloud, or do you see a pattern (a curve, a fan, clusters, a few extreme points)? What does that tell you about your straight-line model?</font>

---

ANSWER

---

<font color=blue>**Question 5.2.**</font> Check your slope and intercept with `np.polyfit` (Lab 8, Part 6). Store `True` or `False` in `agrees`.

In [ ]:
coefficients = np.polyfit(data.column('electronegativity'), data.column('ionization energy (kJ/mol)'), 1)
agrees = np.isclose(coefficients.item(0), my_slope) and np.isclose(coefficients.item(1), my_intercept)
agrees

## 6. The Twist: Which Elements Break the Pattern?
<font color=blue>**Question 6.1.**</font> Add your `errors` array to the table as a column, then sort to find the 5 elements **farthest above** the line and the 5 **farthest below** it. Store the sorted table as `ranked`.

In [ ]:
ranked = data.with_columns('error', errors).select('symbol', 'name', 'category', 'period', 'error').sort('error', descending=True)
ranked.show(5)
ranked.sort('error').show(5)

<font color=blue>**Question 6.2.**</font> Periodic trends are cleanest **across a period**. Fit a line to the elements of **period 2** only (Li through F). Store its correlation as `r_period2`.

In [ ]:
period2 = data.where('period', 2)
r_period2 = correlation(period2, 'electronegativity', 'ionization energy (kJ/mol)')
r_period2

<font color=green>1. Which kinds of elements fall far above the line? (Look at their categories and groups.) Which fall far below?
2. Why might the trend be tighter within a single period?
3. Both properties are driven by how strongly the nucleus holds the outer electrons (effective nuclear charge). Does electronegativity **cause** ionization energy? What does a strong *r* tell you here?</font>

---

ANSWER

---

## 7. Your Final Figure
This is the product of your microproject. Your instructor will show the class gallery on screen, so make a figure a scientist would be proud of. **Requirements:**

- [ ] Scatter plot of the data
- [ ] Your regression line, labeled in the legend with its equation and *r*
- [ ] Axis labels **with units** (e.g. `electronegativity (Pauling)` and `first ionization energy (kJ/mol)`)
- [ ] A title that states your **finding**, not just the variable names
- [ ] **Also** label at least 5 interesting elements with their symbols using `plt.annotate` (for example, the biggest outliers).

Build everything after `fig = plt.figure(...)` so `submit_plot` can save it.

<details><summary><b>💡 Starter pattern</b></summary>

```python
fig = plt.figure(figsize=(9, 5.5))
plt.scatter(x_values, y_values, s=14, alpha=0.6, label='data')
line_x = make_array(low, high)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', lw=2,
         label=f'y = {my_slope:.3f}x + {my_intercept:.1f},  r = {my_r:.2f}')
plt.xlabel('...'); plt.ylabel('...'); plt.title('...'); plt.legend()
plt.show()
```

</details>

In [ ]:
fig = plt.figure(figsize=(9, 5.5))
for per in np.arange(1, 7):
    t = data.where('period', per)
    plt.scatter(t.column('electronegativity'), t.column('ionization energy (kJ/mol)'), s=22, label=f'period {per}')
line_x = make_array(0.7, 4.0)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', lw=2,
         label=f'IE = {my_slope:.0f}·EN {my_intercept:+.0f},  r = {my_r:.2f}')
for sym in make_array('He', 'Ne', 'F', 'Au', 'Cs', 'H'):
    row = data.where('symbol', sym)
    if row.num_rows:
        plt.annotate(sym, (row.column('electronegativity').item(0), row.column('ionization energy (kJ/mol)').item(0)),
                     xytext=(5, 4), textcoords='offset points', fontsize=11, weight='bold')
plt.xlabel('electronegativity (Pauling)'); plt.ylabel('first ionization energy (kJ/mol)')
plt.title('Electronegativity and ionization energy rise together (r = 0.90)')
plt.legend(fontsize=8); plt.show()

<font color=blue>**Question 7.1.**</font> Write a **one-sentence caption** stating your main finding, then submit. You may submit again as often as you like — the newest version replaces the old one.

In [ ]:
caption = 'Electronegativity and first ionization energy are strongly associated (r = 0.90), with the tightest trend across a single period; both reflect effective nuclear charge.'
submit_plot(fig, name, 'MP5', r=my_r, slope=my_slope, intercept=my_intercept, caption=caption)

## 8. Reflection
<font color=green>In 3–5 sentences: What did the data say about your research question? How confident are you, and why? What was hardest to do **without** step-by-step instructions? Be ready to explain your figure to the class in 60 seconds.</font>

---

ANSWER

---

### 🏁 Done!
Save your notebook (💾). Your figure is already in the class gallery. Download and upload an HTML copy to Canvas if your instructor asks for one ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)).